# Multi-Benchmark Biomedical Fusion Research
CAP 6776 • Instructor: Juan D. Yepes • Research direction based on Professor Dingding Wang's feedback.

Enable Internet. Select GPU for full model inference. Python 3.11/3.12 required. This notebook produces new results only after execution. See docs/research-extension.md for the declared comparison family and limitations.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
assert (3,11) <= sys.version_info[:2] < (3,13), "Project requires Python 3.11/3.12"
ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
ROOT.mkdir(parents=True, exist_ok=True)
BRANCH = "research/multibenchmark-fusion-validation"
REPO = ROOT / "biomedical-ir-research"
if not REPO.exists():
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",
        "https://github.com/Arungharami/biomedical-hybrid-ir.git",str(REPO)],check=True)
else:
    active = subprocess.check_output(["git","-C",str(REPO),"branch","--show-current"],text=True).strip()
    assert active == BRANCH, "Use a fresh work folder or the correct branch"
os.chdir(REPO)
print(subprocess.check_output(["git","rev-parse","HEAD"],text=True))
subprocess.run([sys.executable,"-m","pip","install","-r","requirements.txt"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-e","."],check=True)

In [ ]:
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests",
    "-p","test_research_extension.py","-v"],check=True)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_RERANK = False  # Adds seven matched-source reranking comparisons.
RUN_TREC_COVID = False  # Larger benchmark; frozen parameters required.
print("Device:",DEVICE)

## New benchmark runs
Runs use timestamped folders and fresh data/model provenance. The notebook stops on failure; it never substitutes historical or synthetic output.

In [ ]:
def execute(dataset, selection=None):
    before = set(Path("results/research").glob(dataset+"/*"))
    command = [sys.executable,"scripts/run_research.py","--dataset",dataset,"--device",DEVICE]
    if RUN_RERANK: command += ["--rerank"]
    if selection is not None: command += ["--frozen-selection",str(selection)]
    subprocess.run(command,check=True)
    created = set(Path("results/research").glob(dataset+"/*")) - before
    assert len(created)==1, "Expected one fresh run directory"
    folder = created.pop()
    manifest = json.loads((folder/"manifest.json").read_text())
    assert manifest["status"]=="complete"
    return folder
nfcorpus_run = execute("nfcorpus")
scifact_run = execute("scifact")
runs = [nfcorpus_run,scifact_run]

In [ ]:
if RUN_TREC_COVID:
    covid_run = execute("trec-covid", nfcorpus_run/"selection.json")
    runs.append(covid_run)
subprocess.run([sys.executable,"scripts/summarize_research.py",*[str(p) for p in runs]],check=True)

In [ ]:
import pandas as pd
for folder in runs:
    print(folder)
    display(pd.read_csv(folder/"summary.csv"))
summary = json.loads(Path("results/research/study-summary.json").read_text())
display(pd.DataFrame(summary["comparisons"])[["dataset","system_a","system_b",
    "mean_difference","ci95_low","ci95_high","p_holm_study","reject_study_at_05"]])

## Export
Download the zip from Kaggle Outputs. Intervals are pointwise; study p-values are Holm-adjusted. Subgroups are descriptive. Interpret pool-limited MAP/Recall carefully. Review provenance before updating the paper.

In [ ]:
import shutil
print(shutil.make_archive(str(ROOT/"biomedical-multibenchmark-results"),"zip",REPO/"results/research"))